# Solution - Audit and rescue the rent model

Instructor solution. The starter `buggy_rent_model.ipynb` hides **six** methodology
bugs (L04-L07). Below: the filled audit table (Part 1), the corrected pipeline
(Part 2), and the honest model + model card (Part 3).

**Key teaching point:** most of these bugs **barely move the headline number** -
a model can be riddled with methodology errors and still print 0.83. You catch
them by reading the code against the rules, not by hunting for a value that looks
wrong. On this data only **leakage (bug 1)** and **log-units reporting (bug 5)**
actually distort the reported result; the rest are still wrong and would bite on
other data, fixed hyperparameters, or interpretation.

## Part 1 - Audit

| # | Bug (where in the starter) | Rule broken | Why it's wrong (and its real impact here) | Fix |
|---|---|---|---|---|
| **1** | `locality_mean_rent = groupby("Area Locality")["Rent"].transform("mean")` on the **full df**, and `ohe.fit_transform` on the full df, both **before the split** | **L04** data leakage / L01c *fit on train, transform on train+test* | the locality mean is built from test rows' rents (and the row's own rent), so the model peeks at the target. **Real impact: inflates the sealed-test log-R^2 from ~0.80 to 0.83.** (Its coefficient prints as 0.00000 only because the feature is unscaled - it still moves predictions; removing it drops R^2 ~0.03.) | drop the leaky feature (HW2's choice) or target-encode **inside CV folds**; fit every transform on train only, via a `Pipeline` |
| **2** | `score = r2_score(y_test, ...)` inside the alpha sweep | **L04** *the test set is touched once; selection uses validation/CV* | model selection now peeks at the test set, so the test set is no longer an honest hold-out. **Impact here: ~0** - the peeked alpha (1.0) happens to equal the honest CV choice - but on data where alpha matters this silently inflates. A bug you find by *reading*, not by the score | choose alpha by **CV on the training data**; keep the test set sealed until the very end |
| **3** | `Ridge` fit on **unscaled** features | **L05** *always scale before Ridge/Lasso* | the penalty is fair only by luck and the coefficients are uninterpretable (`Size` shows 0.0004 = *units*, not unimportance). **Impact on score here: 0** - once alpha is CV-tuned, Ridge predictions are scale-invariant (demonstrated below) - but the moment regularization actually bites (fixed alpha, stronger penalty, polynomial/correlated features) it silently mangles the fit | `StandardScaler` on the numerics, **inside the pipeline**, before the penalty |
| **4** | alpha grid `[0.1, 0.2, ..., 1.0]` - linear and narrow | **L06** *log-uniform scale; bracket the optimum* | the winner sits at the grid's **boundary** (1.0), so you cannot tell from inside the grid whether the true optimum is there or further out. **Impact here: 0** - widening confirms ~1 - but a boundary winner is always a signal to widen and re-run | search alpha **log-uniform** over a wide range (`1e-3 .. 1e3`); if the winner lands at an edge, widen |
| **5** | reports RMSE/MAE on the **log** target ("0.37"); then compares R^2 across two different y-subsets | **L07** *report in the units that matter; never compare R^2 across different y-ranges* | **Real impact:** RMSE 0.37 is in **log units** - back-transformed it is ~**84,660** rent units, a wildly different story. And R^2 is range-dependent: in the starter, log-R^2 is **0.83** (full) vs **0.76** (normal flats) - the same model looks different just by restricting the range, so the comparison is meaningless (Part 3 shows the same effect on the honest model) | back-transform predictions; report **MAE/MedAE in rent units**; never compare R^2 across ranges |
| **6** | the "FINAL reported score" is a CV score on the (leaked) training matrix | **L04/L06** *tuning inflates the score; report a held-out test once* | there is **no sealed evaluation** anywhere - this CV runs on the leaked training data and reuses the rows that drove selection. Even without the leak, reporting the selection score as the result is optimistic | after CV picks alpha on train, evaluate **once** on the sealed test set (or use nested CV) |

**Bonus catch (no bug #):** there is **no `DummyRegressor` baseline**, so "R^2 = 0.83" has no reference point. Always establish the predict-the-mean floor first (L04).

## Part 2 - The fix

Split first, every transform inside the pipeline (so it refits per CV fold -
no leakage), numerics scaled before the penalty, alpha tuned by CV on the
**training data only** over a wide log range, test set sealed.

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (r2_score, mean_squared_error,
                             mean_absolute_error, median_absolute_error)

SEED = 509
np.random.seed(SEED)

df = pd.read_csv("../01_regression_intro/data/House_Rent_Dataset.csv")

# leaky locality feature is gone; raw columns only - all transforms go in the pipeline
num = ["BHK", "Size", "Bathroom"]
cat = ["City", "Furnishing Status", "Area Type"]
X = df[num + cat]
y_raw = df["Rent"].values
y_log = np.log1p(y_raw)

# split FIRST, before any preprocessing; keep raw Rent aligned for honest reporting
X_tr, X_te, ylog_tr, ylog_te, yraw_tr, yraw_te = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=SEED)
print("train:", X_tr.shape, " test:", X_te.shape, " (test sealed until Part 3)")

train: (3796, 6)  test: (950, 6)  (test sealed until Part 3)


In [2]:
# preprocessing INSIDE the pipeline -> refit on each fold's train rows only (no leakage),
# and numerics scaled before Ridge sees them (fixes bug 3)
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat),
])
pipe = Pipeline([("pre", pre), ("ridge", Ridge())])

# tune alpha by CV on TRAIN ONLY, log-uniform wide range (fixes bugs 2 + 4)
grid = {"ridge__alpha": np.logspace(-3, 3, 13)}
gs = GridSearchCV(pipe, grid, cv=KFold(5, shuffle=True, random_state=SEED),
                  scoring="r2")

# 'Area Type' has a rare level (Built Area) that can be absent from a fold's train
# split; handle_unknown="ignore" then encodes it as zeros (intended), emitting one
# benign UserWarning per fold - silence just that message for a clean log.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Found unknown categories")
    gs.fit(X_tr, ylog_tr)

best = gs.best_params_["ridge__alpha"]
print(f"best alpha = {best:g}   CV R2(log) = {gs.best_score_:.3f}")
print(f"interior to the search range? {1e-3 < best < 1e3}  "
      f"(a boundary winner would mean 'widen and re-run')")

best alpha = 1   CV R2(log) = 0.790
interior to the search range? True  (a boundary winner would mean 'widen and re-run')


### Bug 3 is about interpretability and safety, not score

Two facts worth showing students explicitly: scaling makes the `Size` coefficient
readable, and - because Ridge predictions are scale-invariant once `alpha` is
tuned - it does **not** change the CV score on this data. The rule still holds;
the harm is latent.

In [3]:
# (a) scaling makes the coefficient interpretable (units, not importance)
ridge_fit = gs.best_estimator_.named_steps["ridge"]
feat = (num + list(gs.best_estimator_.named_steps["pre"]
                   .named_transformers_["cat"].get_feature_names_out(cat)))
size_coef = dict(zip(feat, ridge_fit.coef_))["Size"]
print(f"scaled Ridge coef on Size = {size_coef:.4f}  "
      f"(unscaled it was 0.0004 - same model, just units)")

# (b) but the tuned CV score is identical scaled vs unscaled - bug 3 does not move it
def cv_r2(scale):
    steps = [("num", StandardScaler() if scale else "passthrough", num),
             ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat)]
    g = GridSearchCV(Pipeline([("pre", ColumnTransformer(steps)), ("ridge", Ridge())]),
                     grid, cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories")
        g.fit(X_tr, ylog_tr)
    return g.best_score_

print(f"CV R2(log): unscaled={cv_r2(False):.4f}  scaled={cv_r2(True):.4f}  "
      f"-> identical (scale-invariant once tuned)")

scaled Ridge coef on Size = 0.3018  (unscaled it was 0.0004 - same model, just units)


CV R2(log): unscaled=0.7903  scaled=0.7903  -> identical (scale-invariant once tuned)


### Lasso, done right

With scaling and a real `max_iter`, Lasso converges and we can read which
features it keeps vs zeros (the L05 sparsity property).

In [4]:
from sklearn.exceptions import ConvergenceWarning
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    warnings.simplefilter("error", ConvergenceWarning)   # fail loudly ONLY if it does not converge
    lasso = Pipeline([("pre", pre),
                      ("lasso", Lasso(alpha=0.001, max_iter=50_000))]).fit(X_tr, ylog_tr)

coefs = dict(zip(feat, lasso.named_steps["lasso"].coef_))
kept = {k: round(v, 3) for k, v in coefs.items() if abs(v) > 1e-6}
dropped = [k for k, v in coefs.items() if abs(v) <= 1e-6]
print("Lasso converged. kept:", kept)
print("zeroed:", dropped or "(none at this alpha)")

Lasso converged. kept: {'BHK': 0.203, 'Size': 0.3, 'Bathroom': 0.138, 'City_Chennai': -0.064, 'City_Delhi': 0.227, 'City_Hyderabad': -0.183, 'City_Kolkata': -0.317, 'City_Mumbai': 1.121, 'Furnishing Status_Semi-Furnished': -0.155, 'Furnishing Status_Unfurnished': -0.272, 'Area Type_Carpet Area': 0.177}
zeroed: ['Area Type_Super Area']


## Part 3 - Honest model + model card

A `DummyRegressor` floor, then the **single** sealed-test evaluation reported in
rent units, then residual diagnostics that expose where the model fails.

In [5]:
# baseline: predict the mean log-rent (fixes the 'no reference point' gap)
dummy_cv = cross_val_score(DummyRegressor(strategy="mean"), X_tr, ylog_tr,
                           cv=KFold(5, shuffle=True, random_state=SEED),
                           scoring="r2").mean()
print(f"DummyRegressor CV R2(log) = {dummy_cv:.3f}   (our model must clear this)")

DummyRegressor CV R2(log) = -0.001   (our model must clear this)


In [6]:
# THE one-time test evaluation - everything above used train only
p_log = gs.predict(X_te)
p_raw = np.expm1(p_log)

logR2  = r2_score(ylog_te, p_log)
mae    = mean_absolute_error(yraw_te, p_raw)
medae  = median_absolute_error(yraw_te, p_raw)
rmse   = mean_squared_error(yraw_te, p_raw) ** 0.5
print(f"log-R2  = {logR2:.3f}")
print(f"MAE     = {mae:,.0f} rent units")
print(f"MedAE   = {medae:,.0f} rent units   <- robust 'typical error'")
print(f"RMSE    = {rmse:,.0f} rent units   <- inflated by the luxury tail")

log-R2  = 0.799
MAE     = 11,364 rent units
MedAE   = 3,639 rent units   <- robust 'typical error'
RMSE    = 31,807 rent units   <- inflated by the luxury tail


In [7]:
# bug 5b demonstrated: R2 depends on the y-range, so comparing it across ranges is meaningless
lux = yraw_te >= 100_000
print(f"raw-R2 full test           = {r2_score(yraw_te, p_raw):.2f}")
print(f"raw-R2 normal flats (<100k) = {r2_score(yraw_te[~lux], p_raw[~lux]):.2f}")
print("same model, same predictions - only the y-range changed.")

raw-R2 full test           = 0.58
raw-R2 normal flats (<100k) = 0.63
same model, same predictions - only the y-range changed.


### Residual diagnostics: where does it fail?

Predicted-vs-actual (log axes) and residuals-vs-fitted. The model tracks typical
flats but systematically **under-predicts luxury flats** - the long right tail
the log target deliberately de-emphasises.

In [8]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
ARM_RED, ARM_BLUE = "#D90012", "#0033A0"

lim = [yraw_te.min(), yraw_te.max()]
fig = go.Figure()
fig.add_trace(go.Scatter(x=yraw_te, y=p_raw, mode="markers",
                         marker=dict(color=ARM_BLUE, size=4, opacity=0.45),
                         name="predictions"))
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines",
                         line=dict(color=ARM_RED, dash="dash"), name="perfect"))
fig.update_xaxes(type="log", title="actual Rent")
fig.update_yaxes(type="log", title="predicted Rent")
fig.update_layout(title="Predicted vs actual (log axes): luxury flats sit below the line",
                  width=720, height=470)
fig

In [9]:
resid = ylog_te - p_log
fig2 = go.Figure(go.Scatter(x=p_log, y=resid, mode="markers",
                            marker=dict(color=ARM_BLUE, size=4, opacity=0.45)))
fig2.add_hline(y=0, line=dict(color=ARM_RED, dash="dash"))
fig2.update_layout(title="Residuals vs fitted (log scale)",
                   xaxis_title="fitted log-Rent", yaxis_title="residual",
                   width=720, height=420)
fig2

In [10]:
# quantify the failure mode for the model card
print(f"MedAE on normal flats (<100k): {median_absolute_error(yraw_te[~lux], p_raw[~lux]):,.0f}")
print(f"MedAE on luxury flats (>=100k): {median_absolute_error(yraw_te[lux], p_raw[lux]):,.0f}"
      f"   ({lux.sum()} flats)")

MedAE on normal flats (<100k): 3,352
MedAE on luxury flats (>=100k): 59,029   (57 flats)


In [11]:
print("=" * 54)
print(" MODEL CARD - Rent predictor (honest)")
print("=" * 54)
print(f" Data        : House_Rent_Dataset.csv (n={len(df)}), target log1p(Rent)")
print(f" Model       : Ridge(alpha={best:g}) on scaled numerics + OHE categoricals")
print(f" Selection   : 5-fold CV on train; test evaluated ONCE")
print(f" Baseline    : DummyRegressor R2(log)={dummy_cv:.3f}")
print( " Test scores :")
print(f"   log-R2 = {logR2:.3f}   MAE = {mae:,.0f}   MedAE = {medae:,.0f}   RMSE = {rmse:,.0f}")
print(" Headline    : typical error (MedAE) ~ {:,.0f} rent units".format(medae))
print(" Failure mode: under-predicts luxury flats (>=100k); RMSE tail-dominated")
print(" Use MedAE/MAE, not RMSE or raw-R2, as the headline for this skewed target")
print("=" * 54)

 MODEL CARD - Rent predictor (honest)
 Data        : House_Rent_Dataset.csv (n=4746), target log1p(Rent)
 Model       : Ridge(alpha=1) on scaled numerics + OHE categoricals
 Selection   : 5-fold CV on train; test evaluated ONCE
 Baseline    : DummyRegressor R2(log)=-0.001
 Test scores :
   log-R2 = 0.799   MAE = 11,364   MedAE = 3,639   RMSE = 31,807
 Headline    : typical error (MedAE) ~ 3,639 rent units
 Failure mode: under-predicts luxury flats (>=100k); RMSE tail-dominated
 Use MedAE/MAE, not RMSE or raw-R2, as the headline for this skewed target


## Conclusion

After fixing all six bugs, the honest sealed-test **log-R^2 is ~0.80** - a hair
below the teammate's contaminated 0.83, and far above the dummy baseline (~0.0).
The typical error (MedAE) is a few thousand rent units, but the model
**under-predicts luxury flats**, which the log target intentionally
de-emphasises.

The thing to internalise: **only two of the six bugs actually moved the number**
(the leak and the log-units reporting). The other four printed a perfectly
respectable 0.83 while being wrong. That is exactly why you audit the *code*
against the rules - a clean-looking metric is not evidence of clean methodology.

**Bonus directions:** nested CV for a fully unbiased estimate; target-encode
`Area Locality` correctly inside CV folds (instead of dropping it); a P10-P90
prediction interval (quantile regression / conformal); per-city error breakdown.